# Analyse du Global Power Plant Database

Ce notebook applique **NumPy**, **Pandas** et **Matplotlib/Seaborn** à un jeu de données réel sur les centrales électriques du monde entier.

Source officielle du dataset :
- WRI Global Power Plant Database
- Dépôt CSV public associé au projet WRI

Objectifs :
- charger et nettoyer les données,
- explorer les statistiques descriptives,
- analyser la production par type de combustible,
- tester des hypothèses statistiques,
- étudier l’évolution temporelle,
- visualiser les résultats,
- illustrer des opérations matricielles utiles en data science.

## 1) Import des bibliothèques et chargement des données

In [ ]:

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

plt.style.use("default")
sns.set_theme(style="whitegrid")

DATA_URL = "https://raw.githubusercontent.com/wri/global-power-plant-database/master/output_database/global_power_plant_database.csv"
LOCAL_PATH = "global_power_plant_database.csv"

# Chargement depuis l'URL officielle si le fichier local n'existe pas.
try:
    if os.path.exists(LOCAL_PATH):
        df = pd.read_csv(LOCAL_PATH, low_memory=False)
    else:
        df = pd.read_csv(DATA_URL, low_memory=False)
except Exception as e:
    raise RuntimeError(
        "Impossible de charger le dataset. Vérifiez la connexion Internet "
        "ou placez le fichier global_power_plant_database.csv dans le même dossier."
    ) from e

print("Dimensions du dataset :", df.shape)
display(df.head())

## 2) Inspection rapide et nettoyage

In [ ]:

# Aperçu des colonnes
print("Colonnes disponibles :")
print(df.columns.tolist())

# Valeurs manquantes par colonne
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_values": missing,
    "missing_pct": missing_pct
})
display(missing_table.head(15))

# Colonnes attendues les plus utiles
expected_cols = [
    "country", "country_long", "name", "gppd_idnr", "capacity_mw",
    "latitude", "longitude", "primary_fuel", "commissioning_year",
    "owner", "source", "url"
]

existing_cols = [c for c in expected_cols if c in df.columns]
df = df.copy()

# Conversion robuste des colonnes numériques si elles existent
for col in ["capacity_mw", "latitude", "longitude", "commissioning_year"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Retrait des doublons éventuels
df = df.drop_duplicates()

# Stratégie simple de traitement :
# - on conserve les lignes même si quelques champs secondaires sont manquants,
# - on supprime les lignes sans variables essentielles pour les analyses principales.
essential = [c for c in ["primary_fuel", "capacity_mw", "country"] if c in df.columns]
df_clean = df.dropna(subset=essential).copy()

# Imputation légère sur certaines variables numériques pour l'analyse globale
for col in ["latitude", "longitude", "commissioning_year"]:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Dimensions après nettoyage :", df_clean.shape)
display(df_clean[essential + [c for c in ["latitude", "longitude", "commissioning_year"] if c in df_clean.columns]].head())

## 3) Statistiques descriptives avec Pandas et NumPy

In [ ]:

numeric_cols = [c for c in ["capacity_mw", "latitude", "longitude", "commissioning_year"] if c in df_clean.columns]

summary = df_clean[numeric_cols].describe().T
summary["median"] = df_clean[numeric_cols].median(numeric_only=True)
summary["variance"] = df_clean[numeric_cols].var(numeric_only=True)
display(summary)

# Récapitulatif NumPy pur sur la capacité installée
capacity = df_clean["capacity_mw"].to_numpy()
capacity_mean = np.mean(capacity)
capacity_median = np.median(capacity)
capacity_std = np.std(capacity, ddof=1)

print(f"Moyenne capacité (MW) : {capacity_mean:,.2f}")
print(f"Médiane capacité (MW) : {capacity_median:,.2f}")
print(f"Écart-type capacité (MW) : {capacity_std:,.2f}")

## 4) Répartition par pays et par type de combustible

In [ ]:

top_countries = df_clean["country_long"].value_counts().head(10) if "country_long" in df_clean.columns else df_clean["country"].value_counts().head(10)
top_fuels = df_clean["primary_fuel"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_countries.sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Top 10 des pays par nombre de centrales")
axes[0].set_xlabel("Nombre de centrales")
axes[0].set_ylabel("Pays")

top_fuels.sort_values(ascending=False).head(10).plot(kind="bar", ax=axes[1])
axes[1].set_title("Top 10 des combustibles / technologies")
axes[1].set_xlabel("Type de combustible")
axes[1].set_ylabel("Nombre de centrales")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## 5) Analyse statistique de la puissance par type de combustible

In [ ]:

fuel_stats = (
    df_clean.groupby("primary_fuel")["capacity_mw"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)
display(fuel_stats.head(15))

# Visualisation des distributions pour les 8 combustibles les plus fréquents
top8 = df_clean["primary_fuel"].value_counts().head(8).index
subset = df_clean[df_clean["primary_fuel"].isin(top8)]

plt.figure(figsize=(14, 6))
sns.boxplot(data=subset, x="primary_fuel", y="capacity_mw")
plt.title("Distribution de la capacité installée par type de combustible")
plt.xlabel("Type de combustible")
plt.ylabel("Capacité installée (MW)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Test d'hypothèse : la moyenne de capacité diffère-t-elle entre deux combustibles ?
On compare ici les deux types les plus fréquents du dataset avec un **test t de Welch**.

In [ ]:

top2 = df_clean["primary_fuel"].value_counts().head(2).index.tolist()
fuel_a, fuel_b = top2

a = df_clean.loc[df_clean["primary_fuel"] == fuel_a, "capacity_mw"].dropna().to_numpy()
b = df_clean.loc[df_clean["primary_fuel"] == fuel_b, "capacity_mw"].dropna().to_numpy()

t_stat, p_value = stats.ttest_ind(a, b, equal_var=False, nan_policy="omit")

print(f"Comparaison : {fuel_a} vs {fuel_b}")
print(f"t-statistic = {t_stat:.4f}")
print(f"p-value     = {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print("Conclusion : on rejette H0. Les moyennes sont statistiquement différentes.")
else:
    print("Conclusion : on ne rejette pas H0. Pas de différence statistiquement significative détectée.")

### Variante NumPy : test par permutation (approche non paramétrique)

In [ ]:

rng = np.random.default_rng(42)

combined = np.concatenate([a, b])
n_a = len(a)
observed = np.mean(a) - np.mean(b)

n_perm = 5000
perm_stats = np.empty(n_perm)

for i in range(n_perm):
    perm = rng.permutation(combined)
    perm_stats[i] = np.mean(perm[:n_a]) - np.mean(perm[n_a:])

perm_p = np.mean(np.abs(perm_stats) >= np.abs(observed))

print(f"Différence observée des moyennes = {observed:.4f}")
print(f"p-value permutationnelle = {perm_p:.6f}")

## 6) Analyse temporelle
Le dataset contient une année de mise en service (`commissioning_year`) pour une partie des centrales.

In [ ]:

if "commissioning_year" in df_clean.columns:
    year_counts = (
        df_clean["commissioning_year"]
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
    )

    yearly_capacity = (
        df_clean.dropna(subset=["commissioning_year"])
        .assign(commissioning_year=lambda x: x["commissioning_year"].astype(int))
        .groupby("commissioning_year")["capacity_mw"]
        .sum()
        .sort_index()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    year_counts.plot(ax=axes[0])
    axes[0].set_title("Nombre de centrales par année de mise en service")
    axes[0].set_xlabel("Année")
    axes[0].set_ylabel("Nombre")

    yearly_capacity.plot(ax=axes[1])
    axes[1].set_title("Capacité installée totale par année de mise en service")
    axes[1].set_xlabel("Année")
    axes[1].set_ylabel("Capacité totale (MW)")

    plt.tight_layout()
    plt.show()
else:
    print("La colonne commissioning_year n'est pas disponible dans cette version du dataset.")

In [ ]:

if "commissioning_year" in df_clean.columns:
    # Évolution des 5 combustibles les plus fréquents au fil du temps
    fuel_year = (
        df_clean.dropna(subset=["commissioning_year"])
        .assign(commissioning_year=lambda x: x["commissioning_year"].astype(int))
    )

    top5_fuels = fuel_year["primary_fuel"].value_counts().head(5).index
    pivot = (
        fuel_year[fuel_year["primary_fuel"].isin(top5_fuels)]
        .groupby(["commissioning_year", "primary_fuel"])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )

    pivot.plot(figsize=(14, 6))
    plt.title("Évolution du mix énergétique pour les 5 combustibles les plus fréquents")
    plt.xlabel("Année")
    plt.ylabel("Nombre de centrales")
    plt.legend(title="Combustible")
    plt.tight_layout()
    plt.show()

## 7) Visualisation géographique avec latitude et longitude
On représente les centrales sous forme de nuage de points géographique.

In [ ]:

geo_cols = [c for c in ["latitude", "longitude", "capacity_mw", "primary_fuel"] if c in df_clean.columns]
geo_df = df_clean.dropna(subset=[c for c in ["latitude", "longitude"] if c in df_clean.columns]).copy()

# Échantillon pour éviter une figure trop chargée
sample_n = min(10000, len(geo_df))
geo_sample = geo_df.sample(sample_n, random_state=42)

plt.figure(figsize=(12, 7))
plt.scatter(
    geo_sample["longitude"],
    geo_sample["latitude"],
    c=np.log1p(geo_sample["capacity_mw"]),
    s=10,
    alpha=0.4
)
plt.colorbar(label="log(1 + capacité MW)")
plt.title("Distribution géographique des centrales électriques")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

## 8) Opérations matricielles et corrélations
On construit une matrice de covariance sur des variables numériques puis on calcule les valeurs propres et vecteurs propres.

In [ ]:

matrix_cols = [c for c in ["capacity_mw", "latitude", "longitude", "commissioning_year"] if c in df_clean.columns]
X = df_clean[matrix_cols].dropna().to_numpy()

# Standardisation simple avec NumPy
mu = X.mean(axis=0)
sigma = X.std(axis=0, ddof=1)
X_std = (X - mu) / sigma

cov_matrix = np.cov(X_std, rowvar=False)
eigvals, eigvecs = np.linalg.eig(cov_matrix)

cov_df = pd.DataFrame(cov_matrix, index=matrix_cols, columns=matrix_cols)
display(cov_df)

print("Valeurs propres :")
print(np.round(eigvals, 4))

plt.figure(figsize=(8, 6))
sns.heatmap(cov_df, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de covariance standardisée")
plt.tight_layout()
plt.show()

### Interprétation
- Les **valeurs propres** mesurent la quantité de variance capturée par chaque direction.
- Les **vecteurs propres** indiquent les combinaisons linéaires principales des variables.
- Dans ce contexte, ils aident à résumer des relations entre capacité, localisation et année de mise en service, comme dans une analyse en composantes principales.

## 9) Intégration NumPy + Pandas + Matplotlib

Exemples utiles :
- filtrage avancé avec `np.isin`,
- création d'indices booléens NumPy,
- préparation de séries pour les graphiques,
- transformations vectorisées plus rapides que des boucles Python.

In [ ]:

# Exemple de filtrage NumPy + Pandas
selected_fuels = df_clean["primary_fuel"].value_counts().head(3).index.to_numpy()
mask = np.isin(df_clean["primary_fuel"].to_numpy(), selected_fuels)

filtered = df_clean.loc[mask, ["country", "primary_fuel", "capacity_mw"]]
display(filtered.head())

# Exemple de courbe lissée avec moyenne mobile NumPy sur la série temporelle
if "commissioning_year" in df_clean.columns:
    yearly = (
        df_clean.dropna(subset=["commissioning_year"])
        .assign(commissioning_year=lambda x: x["commissioning_year"].astype(int))
        .groupby("commissioning_year")["capacity_mw"]
        .sum()
        .sort_index()
    )

    years = yearly.index.to_numpy()
    values = yearly.to_numpy()

    if len(values) >= 5:
        window = 5
        kernel = np.ones(window) / window
        smoothed = np.convolve(values, kernel, mode="valid")
        smooth_years = years[window - 1:]

        plt.figure(figsize=(12, 5))
        plt.plot(years, values, label="Série brute")
        plt.plot(smooth_years, smoothed, label=f"Moyenne mobile {window} ans")
        plt.title("Capacité installée totale par année avec lissage NumPy")
        plt.xlabel("Année")
        plt.ylabel("Capacité totale (MW)")
        plt.legend()
        plt.tight_layout()
        plt.show()

## 10) Conclusion
Ce notebook montre comment :
- nettoyer un dataset réel avec Pandas,
- accélérer et enrichir les analyses avec NumPy,
- produire des visualisations claires avec Matplotlib et Seaborn,
- appliquer un test d'hypothèse,
- interpréter des matrices de covariance et les valeurs propres.

Vous pouvez maintenant l'exécuter, ajuster les filtres, ou approfondir l'analyse par région, décennie, ou technologie de production.